# html_utils Library Demo
This notebook demonstrates the core features of the `html_utils` library, designed for building complex, hierarchical HTML structures directly in Python.

### Prerequisites
Ensure the `html_utils` library folder is in the same directory as this notebook. This demo requires `plotly` and `ipython`.

In [1]:
from IPython.display import HTML as NotebookHTML, display
import plotly.graph_objects as go

# Core document management
from tts_html_utils.core.compiler import HtmlCompiler

# Text and basic components
from tts_html_utils.core.components import (
    H1, H2, H3, P, Div, Link, HR, BR
)

# Specialized Components
from tts_html_utils.core.components.list import PowerList
from tts_html_utils.core.components.table import PowerTable
from tts_html_utils.core.components.structure import PaneContainer
from tts_html_utils.core.components.plot import ScatterPlot
from tts_html_utils.visdiff.visdiff import VisualDiff

## 1. Basic Text and Generic Components
Standard text components like `H1` and `P` allow for direct style dictionaries. The `Link` component supports opening in new tabs by default.

In [2]:
#children can be added with the children kwarg (used here as position since it's always first)
div = Div(H1("Main Page Header")) 

# or they can be added with the add_child method
div.add_child(P("Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat."))
div.add_child(H2("Section Sub-header"))
div.add_child(P("Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum."))

#example of multiple nested children added in a more complex way
link_example = Link("Click here to visit our website", href="https://example.com")
div.add_child(P(["Find more information by following this link: ", link_example]))

display(NotebookHTML(div.render()))

## 2. Building Hierarchies with `PowerList`
The **PowerList** component provides an "outline-style" builder for nested lists, using `.up_level()` and `.down_level()` to handle tag nesting automatically instead of manual `<ul>` and `<li>` management.

In [3]:
pl = PowerList(ordered=True)  # Renders as <ol>
pl.line("Phase 1: Environment Setup")
pl.up_level(label="Sub-tasks") # Creates a nested list with a label
pl.line("Install Dependencies")
pl.line("Configure Plotly libraries")
pl.down_level()                # Returns to the root level
pl.line("Phase 2: Data Execution")

#every HtmlComponent has a render method, so no need to wrap in a div
display(NotebookHTML(pl.render()))


## 3. Data Visualization with `PowerTable`
**PowerTable** handles the heavy lifting of table creation, including headers, alternating row colors, and sticky headers. It can also inject custom JavaScript for **local filtering** and **local sorting**. Note that two copies are made here as the PowerTable JavaScript has not yet been deconflicted with the Jupyter environment. One instance below has filtering and sorting enabled. The other does not. The one with both enabled will be written to a local HTML file below. The one without can be rendered here safely.

In [4]:
mission_data = [
    {'Project': 'Artemis', 'Budget': '$35B', 'Status': 'Active'},
    {'Project': 'James Webb', 'Budget': '$10B', 'Status': 'Operational'},
    {'Project': 'Mars 2020', 'Budget': '$2.7B', 'Status': 'Operational'}
]


table_no_js = PowerTable(
    row_data=mission_data, 
)
table_no_js.add_header() 
table_no_js.add_superheader("Mission Summary", collapsible=True)

table = PowerTable(
    row_data=mission_data, 
    add_filters='local', # Injects local JS filtering
    add_sorting='local',  # Injects local JS sorting
    id='table-demo'
)
table.add_header() 
table.add_superheader("Interactive Mission Summary", collapsible=True)
display(NotebookHTML(table_no_js.render()))



## 4. Interactive Tabs and Plotly Integration
**PaneContainer** stacks components into a tabbed interface, automatically generating a navigation bar based on pane names. **ScatterPlot** wraps Plotly figures into HTML components.

In [5]:
fig = go.Figure(data=[go.Bar(y=[2, 3, 1], marker_color='navy')])
plot_comp = ScatterPlot(fig=fig, title="Project Analytics")

tabs = PaneContainer()
tabs.add_pane(div, "Basic Text and Generic Components")
tabs.add_pane(pl, "PowerList")
tabs.add_pane(table, "Mission Table")

## 5. Compilation and Final Display
The **HtmlCompiler** acts as the orchestrator, harvesting all required styles and scripts from nested components to build a single document.

In [6]:
compiler = HtmlCompiler(title="Full Library Demo")
compiler.add_body_component(H1('TTS HTML Utils Demo'))
compiler.add_body_component(P('High level demonstration of our workhorse HTML library'))
compiler.add_body_component(HR())
compiler.add_body_component(H2("Interactive Tabs"))
compiler.add_body_component(tabs)

# Render and Display
full_html = compiler.render_to_file('full_demo.html')
